In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from mne.decoding import CSP
import mne
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut
from pickle import dump, load 
import sys
import os
import mlflow
import mlflow.sklearn
from pathlib import Path

sys.path.append(os.path.abspath(os.path.join('..')))

from src.train_BCICIV import train_BCICIV
from src.load_data_BCICIV import load_all_subjects
from src.preprocess import laplacian_filter, channel_aggregation


mne.set_log_level('WARNING')

## LOAD TRAIN DATA FOR BCICIV DATASET

In [2]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A01T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A02T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: ../datasets/BCICIV_2a_gdf/A07T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A08T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A09T.gdf


## CSP + LDA MODEL

In [3]:
mlflow.set_experiment("BCI_Motor_Imagery_Classification_Experiment")

2026/02/24 10:14:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/24 10:14:26 INFO mlflow.store.db.utils: Updating database tables
2026/02/24 10:14:27 INFO mlflow.tracking.fluent: Experiment with name 'BCI_Motor_Imagery_Classification_Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Volumes/MyPassport/EEGNet-Motor-Imagery-BCI/notebooks/mlruns/1', creation_time=1771924467715, experiment_id='1', last_update_time=1771924467715, lifecycle_stage='active', name='BCI_Motor_Imagery_Classification_Experiment', tags={}, workspace='default'>

In [4]:
pipe1 = Pipeline([
    ('csp', CSP(n_components=4, log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

param_grid1 = {
    # 'csp__n_components': [2, 4, 6],
    # 'csp__reg': [None, 'ledoit_wolf', 'oas'],
    # 'clf__solver': ['svd', 'lsqr', 'eigen']
}

In [5]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="LDA_with_CSP"):

    model, params, best_score = train_BCICIV(data, pipe1, loso, param_grid1)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max singular value)
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max singular value)
    Using tolerance 9.7e-05 (2.2e-16 eps * 11 dim * 4e+10  max singular value)
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max singular value)
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max singular value)
    Using tolerance 9.4e-05 (2.2e-16 eps * 11 dim * 3.8e+10  max singular value)
    Estimated rank (data): 11
    data: rank 11 computed from 11 data channels with 0 projectors
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max singular value)
    Using toleranc

2026/02/24 10:14:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/24 10:14:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow register completed: 0.4641378068776769


## CSP + SVM MODEL

In [6]:
pipe2 = Pipeline([
    ('csp', CSP(n_components=4, log=True, norm_trace=False)),
    ('clf', SVC(C=1.0, kernel='rbf'))
])

param_grid2 = {
    # 'csp__n_components': [2, 4, 6],
    # 'csp__reg': [None, 'ledoit_wolf', 'oas'],
    # 'clf__C': [0.1, 1, 10],
    # 'clf__gamma': ['scale', 'auto']
}

In [7]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="SVM_with_CSP"):

    model, params, best_score = train_BCICIV(data, pipe2, loso, param_grid2)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max singular value)
    Estimated rank (data): 11
    data: rank 11 computed from 11 data channels with 0 projectors
    Using tolerance 9.8e-05 (2.2e-16 eps * 11 dim * 4e+10  max singular value)
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max singular value)
    Estimated rank (data): 11
    data: rank 11 computed from 11 data channels with 0 projectors
    Estimated rank (data): 11
    data: rank 11 computed from 11 data channels with 0 projectors
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max singular value)
    Using tolerance 9.6e-05 (2.2e-16 eps * 11 dim * 3.9e+10  max si

2026/02/24 10:15:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/24 10:15:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow register completed: 0.4548606650651376


## CHANNEL AGGREGATION

In [8]:
data_agg = channel_aggregation(data['X'])

## CSP + LDA MODEL

In [9]:
pipe1 = Pipeline([
    ('csp', CSP(n_components=4, log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

param_grid1 = {
    # 'csp__n_components': [2, 4, 6],
    # 'csp__reg': [None, 'ledoit_wolf', 'oas'],
    # 'clf__solver': ['svd', 'lsqr', 'eigen']
}

In [10]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="LDA_with_CSP_data_agg"):
    data_agg_dict = {'X': data_agg, 'y': data['y'], 'subject_ids': data['subject_ids']}
    model, params, best_score = train_BCICIV(data_agg_dict, pipe1, loso, param_grid1)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 2.6e-06 (2.2e-16 eps * 1 dim * 1.2e+10  max singular value)
    Estimated rank (data): 1
    data: rank 1 computed from 1 data channel with 0 projectors
    Using tolerance 2.7e-06 (2.2e-16 eps * 1 dim * 1.2e+10  max singular value)
    Estimated rank (data): 1
    data: rank 1 computed from 1 data channel with 0 projectors
    Using tolerance 2.6e-06 (2.2e-16 eps * 1 dim * 1.2e+10  max singular value)
Computing rank from data with rank=None
Reducing data rank from 1 -> 1
    Estimated rank (data): 1
    data: rank 1 computed from 1 data channel with 0 projectors
Estimating class=0 covariance using EMPIRICAL
Reducing data rank from 1 -> 1
Done.
Estimating class=0 covariance using EMPIRICAL
D

2026/02/24 10:15:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/24 10:15:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow register completed: 0.38737790534268435


## CSP + SVM MODEL

In [11]:
pipe2 = Pipeline([
    ('csp', CSP(n_components=4, log=True, norm_trace=False)),
    ('clf', SVC(C=1.0, kernel='rbf'))
])

param_grid2 = {
    # 'csp__n_components': [2, 4, 6],
    # 'csp__reg': [None, 'ledoit_wolf', 'oas'],
    # 'clf__C': [0.1, 1, 10],
    # 'clf__gamma': ['scale', 'auto']
}

In [12]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="SVM_with_CSP_data_agg"):
    data_agg_dict = {'X': data_agg, 'y': data['y'], 'subject_ids': data['subject_ids']}
    model, params, best_score = train_BCICIV(data_agg_dict, pipe2, loso, param_grid2)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 2.7e-06 (2.2e-16 eps * 1 dim * 1.2e+10  max singular value)
    Using tolerance 2.7e-06 (2.2e-16 eps * 1 dim * 1.2e+10  max singular value)
    Using tolerance 2.6e-06 (2.2e-16 eps * 1 dim * 1.2e+10  max singular value)
    Using tolerance 2.6e-06 (2.2e-16 eps * 1 dim * 1.2e+10  max singular value)
    Estimated rank (data): 1
    data: rank 1 computed from 1 data channel with 0 projectors
    Estimated rank (data): 1
    data: rank 1 computed from 1 data channel with 0 projectors
    Estimated rank (data): 1
    data: rank 1 computed from 1 data channel with 0 projectors
    Estimated rank (data): 1
    data: rank 1 computed from 1 data channel with 0

2026/02/24 10:15:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/24 10:15:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow register completed: 0.42126671981456015
